# cyclone_jax — run an experiment

Kernel: **Python (jrt)** (`python -m ipykernel install --user --name jrt`).

Rules this notebook encodes (do not reorder):
1. The FIRST code cell pins the GPU and paths **before any jax import** — JAX initialises once per kernel; restart the kernel to switch GPU.
2. Fresh `run_dir` per run (never reuse — orbax overwrite + OneDrive locks on Windows).
3. Restart the kernel between big runs — device memory accumulates.

In [ ]:
# --- FIRST CELL: site pinning, BEFORE any jax import -----------------
import os, sys
from pathlib import Path

os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')       # pick your GPU
os.environ.setdefault('CYCLONE_JAX_ROOT', '/data/Caribbean-Obs')  # library
os.environ.setdefault('CYCLONE_JAX_NUM_WORKERS', '8')    # ~cores; 0 on Windows
# os.environ['WANDB_API_KEY'] = '...'   # or wandb login in the shell

REPO = Path.cwd()
while not (REPO / 'jrt').exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'jrt'))
print('repo:', REPO)
print('gpu :', os.environ['CUDA_VISIBLE_DEVICES'],
      '| root:', os.environ['CYCLONE_JAX_ROOT'],
      '| workers:', os.environ['CYCLONE_JAX_NUM_WORKERS'])

## Pick the experiment

The entry yaml points at one data scenario + one model. Scenario variants: `train` / `train_land` / `train_marine`, `memorise` / `memorise_land` / `memorise_marine`.

In [ ]:
from experiments.cyclone_jax.config import load_config, CONFIG_DIR

ENTRY = CONFIG_DIR / 'train' / 'train.yaml'   # edit data:/model: pointers there
cfg = load_config(ENTRY)
print('data :', cfg['names']['data'], '| model:', cfg['names']['model'])
print('runs to run_dir:', cfg['trainer'].get('run_dir'), '(set a FRESH one per run)')

## Inspect the data (optional but cheap)

In [ ]:
from experiments.cyclone_jax.data.interface import build_data

data = build_data(cfg['data'], seed=cfg['trainer'].get('seed', 0))
for name, idx in data.splits.items():
    print(f'{name:6s} {len(idx):6d} fixes')
batch = next(iter(data.streams['train']))
{k: getattr(v, 'shape', type(v).__name__) for k, v in batch['X'].items()}

## Memorisation ceiling (memorise scenarios)

Run BEFORE the capacity ladder: identical inputs with different labels cannot both be memorised — `max_accuracy` is the ceiling, and separates CANNOT-memorise from DID-NOT-memorise.

In [ ]:
from experiments.cyclone_jax.data.identifiability import input_collisions

report = input_collisions(data.loader, data.splits['train'])
print({k: report[k] for k in ('n_fixes', 'n_unique_inputs',
                              'n_unmemorisable', 'max_accuracy')})
report['conflicts'][:3]   # which storms/times share inputs but not labels

## Train

`main` prints the startup banner (splits, norm bounds, param count, architecture table), writes `run_dir/norm_stats.json` + `data_manifest.json`, runs fit → test → end-of-run figures, and returns `(trainer, test_metrics)`.

In [ ]:
from experiments.cyclone_jax.train.train import main

trainer, test_metrics = main(ENTRY)
test_metrics

## Where the results live

In [ ]:
run_dir = Path(cfg['trainer']['run_dir'])
print('best metric:', trainer._best_metric_value)
sorted(p.name for p in (run_dir / 'figures').glob('*')) if (run_dir / 'figures').exists() else 'no figures yet'